# Notebook 01 — Activation Scale Calibration for Qwen2.5-Coder

This is the foundation of every other notebook in the project. SmoothQuant needs, for each `nn.Linear` in the model, a per-input-channel estimate of how large the activations get. We compute this by running the model on a calibration corpus and tracking the per-channel absolute maximum.

The output is a dict `{module_path: torch.Tensor of shape [in_features]}` saved to `act_scales/qwen25-coder-<size>.pt`. Every downstream notebook loads this file.

## What this notebook does

1. Loads Qwen2.5-Coder-Instruct (7B or 14B) in bf16
2. Hooks every `nn.Linear` to record per-channel activation absmax
3. Feeds 512 samples (512 tokens each) from the Pile validation set
4. Saves the resulting scales dict
5. Sanity-checks the output: prints the most outlier-heavy channels and plots the distribution

## Runtime

- **7B on L4 or A100**: ~8 min
- **14B on A100 40GB**: ~15 min
- 14B on L4: will OOM because bf16 weights alone need ~28 GB

## Why calibrate on Pile (not on code)?

The SmoothQuant paper uses 512 random sentences from the Pile validation split. We use the same dataset to stay faithful to the paper. If you want to experiment with code-specific calibration (e.g., `bigcode/the-stack-smol`), change `DATASET_NAME` below — this is a good ablation for the project's stretch goals.

## Prerequisites

- Repo cloned with `external/smoothquant` subdirectory (`pip install -e external/smoothquant` is NOT strictly needed — we only import our own adapted code — but install it anyway for the upstream reference utilities).
- HuggingFace login: `huggingface-cli login`
- GPU with enough VRAM: 16+ GB for 7B bf16, 32+ GB for 14B bf16

## Section 1 — Setup

In [ ]:
# If running on Colab, mount Drive and cd into the project so we read/write
# to persistent storage. Skip this cell if running locally.
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/qwen-smoothquant-project'
    assert os.path.exists(PROJECT_ROOT), (
        f'Project folder not found at {PROJECT_ROOT}. Upload the repo to Drive first.'
    )
    %cd $PROJECT_ROOT
except ImportError:
    # Running locally - assume we're in the project root already
    PROJECT_ROOT = os.getcwd()
    assert os.path.exists('src/qwen_smooth.py'), (
        'Run this notebook from the qwen-smoothquant-project root.'
    )
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# Install dependencies — only run once per Colab session
!pip install -q transformers accelerate datasets tqdm

In [ ]:
# Make src/ importable
import sys
if 'src' not in sys.path[0]:
    sys.path.insert(0, os.path.abspath('.'))

from src.calibrate import get_act_scales, save_act_scales

In [ ]:
# Configuration — change MODEL_SIZE to '7B' or '14B'
MODEL_SIZE = '7B'                   # '7B' or '14B'
MODEL_ID   = f'Qwen/Qwen2.5-Coder-{MODEL_SIZE}-Instruct'
DATASET_NAME = 'mit-han-lab/pile-val-backup'  # same as SmoothQuant paper
NUM_SAMPLES = 512
SEQ_LEN     = 512
OUT_PATH    = f'act_scales/qwen25-coder-{MODEL_SIZE.lower()}.pt'

print(f'Model:    {MODEL_ID}')
print(f'Dataset:  {DATASET_NAME}')
print(f'Samples:  {NUM_SAMPLES} x {SEQ_LEN} tokens')
print(f'Output:   {OUT_PATH}')

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader

## Section 2 — Load the model

Strictly bf16. No quantization at calibration time — we need the true activation statistics.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f'Loading {MODEL_ID} in bf16 ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
model.eval()
print('Ready.')

In [ ]:
# Log the model architecture — this is useful to have printed in your writeup.
# Note that Qwen2Attention has bias on q/k/v/ proj (unlike Llama) and uses GQA.
cfg = model.config
total_params = sum(p.numel() for p in model.parameters())
print(f'{MODEL_ID}')
print(f'  hidden_size         : {cfg.hidden_size}')
print(f'  intermediate_size   : {cfg.intermediate_size}')
print(f'  num_hidden_layers   : {cfg.num_hidden_layers}')
print(f'  num_attention_heads : {cfg.num_attention_heads}')
print(f'  num_key_value_heads : {cfg.num_key_value_heads}  (GQA ratio: {cfg.num_attention_heads // cfg.num_key_value_heads}x)')
print(f'  vocab_size          : {cfg.vocab_size}')
print(f'  total parameters    : {total_params/1e9:.2f}B')

## Section 3 — Run calibration

`get_act_scales` hooks every `nn.Linear` in the model and records `|input|.max(dim=-1)` — the per-channel absolute maximum over the calibration corpus. This is exactly what the SmoothQuant paper does (see `smoothquant/calibration.py` in the upstream repo).

In [ ]:
act_scales = get_act_scales(
    model=model,
    tokenizer=tokenizer,
    dataset_name=DATASET_NAME,
    num_samples=NUM_SAMPLES,
    seq_len=SEQ_LEN,
)
print(f'Collected scales for {len(act_scales)} Linear modules')

In [ ]:
# Save to Drive/disk before we look at anything — calibration is the most
# expensive step and we don't want to redo it.
os.makedirs('act_scales', exist_ok=True)
save_act_scales(act_scales, OUT_PATH)

## Section 4 — Sanity-check the scales

Three quick checks. If any of these look wrong, don't proceed to quantization until you've understood why.

1. **We have scales for the expected Linears.** For an N-layer Qwen2, we expect `7 * N + 1` Linears: q/k/v/o + gate/up/down per layer, plus the `lm_head`.
2. **Outlier-heavy channels exist.** The paper's key insight is that activation outliers are concentrated in a few channels. We should see 2-5 channels per layer with values much larger than the rest.
3. **Tensor shapes match the Linear input dimensions.** e.g. `q_proj` expects `hidden_size` input channels.

In [ ]:
# Check 1: expected module count
expected_decoder_linears = 7 * cfg.num_hidden_layers
expected_total = expected_decoder_linears + 1  # +1 for lm_head
print(f'Expected Linears in decoder: {expected_decoder_linears}')
print(f'Expected total (incl lm_head): {expected_total}')
print(f'Actually calibrated          : {len(act_scales)}')
assert len(act_scales) == expected_total, (
    'Unexpected number of calibrated Linears. Did a hook fail? Did the model '
    'architecture change?'
)

In [ ]:
# Check 2: outlier distribution — pick a mid-stack layer and show the
# per-channel absmax for q_proj. If the paper's premise holds, we should
# see a few channels dominating.

import matplotlib.pyplot as plt

sample_layer = cfg.num_hidden_layers // 2
sample_key = f'model.layers.{sample_layer}.self_attn.q_proj'
scales = act_scales[sample_key].float().numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(scales)
ax1.set_xlabel('channel index')
ax1.set_ylabel('|activation| max')
ax1.set_title(f'q_proj activation absmax — layer {sample_layer}')
ax1.set_yscale('log')
ax1.grid(alpha=0.3)

ax2.hist(scales, bins=80, log=True)
ax2.set_xlabel('|activation| max')
ax2.set_ylabel('channel count (log)')
ax2.set_title('Distribution of channel absmax values')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('results/plots/activation_outliers_qkv.png', dpi=120, bbox_inches='tight')
plt.show()

top5 = scales.argsort()[-5:][::-1]
print(f'\nTop-5 outlier channels in layer {sample_layer} q_proj:')
for idx in top5:
    print(f'  channel {idx:5d}: {scales[idx]:.3f}')
print(f'Median channel value: {sorted(scales)[len(scales)//2]:.3f}')
print(f'Ratio top-1 / median: {scales.max() / sorted(scales)[len(scales)//2]:.1f}x')

In [ ]:
# Check 3: shape consistency
import torch.nn as nn
mismatches = []
for name, module in model.named_modules():
    if not isinstance(module, nn.Linear):
        continue
    if name not in act_scales:
        mismatches.append((name, 'MISSING'))
        continue
    expected = module.in_features
    got = act_scales[name].numel()
    if expected != got:
        mismatches.append((name, f'expected {expected}, got {got}'))

if mismatches:
    print('FAILURES:')
    for n, msg in mismatches[:10]:
        print(f'  {n}: {msg}')
else:
    print(f'All {len(act_scales)} act_scales have correct shapes.')

## Section 5 — Save summary statistics for the report

The report will want concrete numbers about activation outliers. Save them alongside the scales so notebook 06 (profiling) can reuse them.

In [ ]:
import json

# For each Linear type, compute the mean outlier ratio across all layers.
# Outlier ratio = max(|x|) / median(|x|) per channel.
proj_types = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
              'gate_proj', 'up_proj', 'down_proj']
summary = {}
for pt in proj_types:
    ratios = []
    for name, vec in act_scales.items():
        if name.endswith('.' + pt):
            v = vec.float().numpy()
            med = sorted(v)[len(v) // 2]
            if med > 0:
                ratios.append(float(v.max() / med))
    summary[pt] = {
        'num_layers_calibrated': len(ratios),
        'mean_outlier_ratio': sum(ratios) / len(ratios) if ratios else 0.0,
        'max_outlier_ratio' : max(ratios) if ratios else 0.0,
    }

print(json.dumps(summary, indent=2))

os.makedirs('results', exist_ok=True)
with open(f'results/outlier_summary_{MODEL_SIZE.lower()}.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\nSaved summary to results/outlier_summary_{MODEL_SIZE.lower()}.json')

## Done!

### What you should see if this worked correctly

- **Top-1 / median ratio** in the tens to hundreds for `q_proj`, `k_proj`, `v_proj`, `gate_proj`, `up_proj`. This is the activation-outlier phenomenon the paper identifies.
- **Low outlier ratio** (maybe 2-5x) for `o_proj` and `down_proj` — these take already-processed activations and have less pathological distributions.
- **512 samples** is enough to stabilize the per-channel max within a few percent. You can verify by re-running with 256 samples and comparing.

### Next

→ `02_smooth_and_fakequant.ipynb` — applies the scales we just computed to produce a smoothed model, then W8A8-fake-quantizes it. Measures perplexity on WikiText to verify accuracy is preserved before we spend time on the full code benchmarks.

### Troubleshooting

- **OOM during calibration:** drop `SEQ_LEN` to 256, or use `device_map='auto'` with some layers on CPU (much slower).
- **Pile dataset not downloading:** if `mit-han-lab/pile-val-backup` is gated, try `'NeelNanda/pile-10k'` or `'cerebras/SlimPajama-627B'` (streaming).
- **Calibration only found `N < expected` Linears:** check whether the tokenizer's chat template is adding something unexpected, or whether the model is actually Qwen2 (not Qwen2-VL or Qwen-MoE).
- **Scales are identical across layers:** bug — the hooks probably failed. Check that all hooks in the `finally` block are being removed correctly.